# Diagonal-inertia and friction-model diagnostic

Validates the diagonal-inertia approximation and the viscous-plus-Coulomb friction model. Off-diagonal correlation analysis and R-squared improvement from including the friction term.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py

warnings.filterwarnings("ignore")
np.random.seed(42)

# Paths
ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")
os.makedirs(OUT, exist_ok=True)
OUT_CSV = os.path.join(OUT, "NB12_R1_diagonal_inertia_friction_diagnostic.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0, 0, 0.05])
GRAVITY      = np.array([0, 0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3]
                 for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = (Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6 \
                 else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3]
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

# Registry (healthy only, four tasks)
REGISTRY = {
    "T1_healthy": ("T1_PickPlace/Healthy",   "UR5_T1_healthy_180cyc_*.h5",        "T1"),
    "T2_healthy": ("T2_Assembly/Healthy",    "UR5_T2_healthy_180cyc_*.h5",        "T2"),
    "T3_healthy": ("T3_Palletize/Healthy",   "UR5_T3_healthy_183cyc_*.h5",        "T3"),
    "T4_healthy": ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5","T4"),
}

def load_healthy_cycles_by_task():
    by_task = {t: [] for t in TASK_PAYLOAD}
    for key, (subdir, pattern, task) in REGISTRY.items():
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING  Not found: {key}")
            continue
        with h5py.File(matches[0], "r") as f:
            cnum   = f["cycle_number"][:].astype(int).ravel()
            cur_a  = f["actual_current"][:]
            q_a    = f["actual_q"][:]
            qd_a   = f["actual_qd"][:]
        for c in np.unique(cnum[cnum > 0]):
            mask = cnum == c
            if mask.sum() >= MIN_SAMP:
                by_task[task].append({
                    "q":       q_a[mask],
                    "qd":      qd_a[mask],
                    "current": cur_a[mask],
                })
    return by_task

def build_design_matrices(cycles, task):
    """Build per-joint PSR design matrix Phi (5 cols) and current I.
    Also return per-sample full q-dot and q-double-dot vectors for the
    off-diagonal proxy diagnostic."""
    payload = TASK_PAYLOAD[task]
    Phi   = {j: [] for j in range(6)}
    I_arr = {j: [] for j in range(6)}
    qd_full_rows  = []
    qdd_full_rows = []
    for cyc in cycles:
        q_a = cyc["q"]; qd_a = cyc["qd"]; cur = cyc["current"]
        N = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            qdd_t = np.zeros(6)
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < N-1 else 0.0)
                qdd_t[j] = qdd_j
                Phi[j].append(np.array([
                    tau_g[j], qd_a[t, j], np.sign(qd_a[t, j]),
                    qdd_j, 1.0]))
                I_arr[j].append(cur[t, j])
            qd_full_rows.append(qd_a[t].copy())
            qdd_full_rows.append(qdd_t)
    return (
        {j: np.array(Phi[j])   for j in range(6)},
        {j: np.array(I_arr[j]) for j in range(6)},
        np.array(qd_full_rows),
        np.array(qdd_full_rows),
    )

def fit_per_joint_ols(Phi, I_arr):
    w = {}
    for j in range(6):
        ww, _, _, _ = np.linalg.lstsq(Phi[j], I_arr[j], rcond=None)
        w[j] = ww
    return w

def r2_wright(y_true, y_pred):
    """R^2 = 1 - SS_res / SS_tot. Returns NaN if SS_tot == 0."""
    if len(y_true) == 0:
        return np.nan
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true))**2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

# MAIN
print("=" * 65)
print("NB12 -- Diagonal-inertia and friction-model diagnostic (R1.1)")
print("=" * 65)

print("\n[Step 1] Loading healthy cycles by task...")
by_task = load_healthy_cycles_by_task()
for t, cycs in by_task.items():
    print(f"  {t}: {len(cycs)} healthy cycles")

print("\n[Step 2] Per-task fit and diagnostics (in-sample)...")
results = []

for task in ["T1", "T2", "T3", "T4"]:
    cycs = by_task[task]
    if not cycs:
        continue
    t0 = time.perf_counter()
    Phi, I_arr, qd_full, qdd_full = build_design_matrices(cycs, task)
    w = fit_per_joint_ols(Phi, I_arr)
    fit_time = time.perf_counter() - t0
    n_samp = Phi[0].shape[0]
    print(f"  {task}: {len(cycs)} cycles, {n_samp:,} subsampled samples, "
          f"fit + design in {fit_time:.1f}s")

    for j in range(6):
        i_meas = I_arr[j]
        i_pred = Phi[j] @ w[j]
        resid  = i_meas - i_pred

        abs_qd  = np.abs(qd_full[:, j])
        abs_qdd = np.abs(qdd_full[:, j])

        # (A) Operating envelope
        env = dict(
            qd_p50  = float(np.percentile(abs_qd,  50)),
            qd_p95  = float(np.percentile(abs_qd,  95)),
            qd_p99  = float(np.percentile(abs_qd,  99)),
            qd_max  = float(abs_qd.max()),
            qdd_p50 = float(np.percentile(abs_qdd, 50)),
            qdd_p95 = float(np.percentile(abs_qdd, 95)),
            qdd_p99 = float(np.percentile(abs_qdd, 99)),
            qdd_max = float(abs_qdd.max()),
        )

        # (B) Regime-split R^2 (top vs bottom |q-dot| quartile)
        q_lo = np.percentile(abs_qd, 25)
        q_hi = np.percentile(abs_qd, 75)
        mask_lo = abs_qd <= q_lo
        mask_hi = abs_qd >= q_hi
        r2_full = r2_wright(i_meas, i_pred)
        r2_lo   = r2_wright(i_meas[mask_lo], i_pred[mask_lo])
        r2_hi   = r2_wright(i_meas[mask_hi], i_pred[mask_hi])

        # (C) Zero-crossing residual
        zc_thr = 0.05 * env["qd_p99"] if env["qd_p99"] > 0 else 1e-6
        mask_zc = abs_qd < zc_thr
        rms_typical = float(np.sqrt(np.mean(resid**2)))
        if mask_zc.sum() >= 10:
            rms_zc = float(np.sqrt(np.mean(resid[mask_zc]**2)))
            zc_ratio = (rms_zc / rms_typical) if rms_typical > 0 else np.nan
        else:
            rms_zc = np.nan
            zc_ratio = np.nan
        n_zc = int(mask_zc.sum())

        # (D) Off-diagonal inertia / Coriolis proxy
        # sum_{k != j} |q-dot_j * q-dot_k|
        off_proxy = np.zeros_like(qd_full[:, j])
        for k in range(6):
            if k != j:
                off_proxy = off_proxy + np.abs(qd_full[:, j] * qd_full[:, k])
        if off_proxy.std() > 0:
            r_off = float(np.corrcoef(resid, off_proxy)[0, 1])
        else:
            r_off = np.nan

        results.append(dict(
            task              = task,
            joint             = f"J{j+1}",
            n_samples         = n_samp,
            n_zerocross       = n_zc,
            **env,
            r2_full           = r2_full,
            r2_low_qd         = r2_lo,
            r2_high_qd        = r2_hi,
            r2_high_minus_low = (r2_hi - r2_lo) if not (np.isnan(r2_hi) or np.isnan(r2_lo)) else np.nan,
            resid_rms_typical = rms_typical,
            resid_rms_zerocross = rms_zc,
            zerocross_ratio   = zc_ratio,
            off_diag_corr     = r_off,
        ))

df = pd.DataFrame(results)
df.to_csv(OUT_CSV, index=False)
print(f"\nWrote: {OUT_CSV}")
print(f"Rows:  {len(df)} ({df['task'].nunique()} tasks x {df['joint'].nunique()} joints)")

# Headline summary
print("\n" + "=" * 65)
print("Summary for R1.1 paragraph")
print("=" * 65)

print("\n(A) Operating envelope: p99 |q-dot| (rad/s) and p99 |q-double-dot| (rad/s^2)")
piv = df.pivot(index="joint", columns="task", values="qd_p99")
print("\np99 |q-dot|:")
print(piv.to_string(float_format=lambda x: f"{x:.2f}"))
piv = df.pivot(index="joint", columns="task", values="qdd_p99")
print("\np99 |q-double-dot|:")
print(piv.to_string(float_format=lambda x: f"{x:.2f}"))

print("\n(B) Regime-split R^2 (top vs bottom |q-dot| quartile)")
print("Mean across joints, per task:")
print(df.groupby("task")[["r2_low_qd", "r2_high_qd", "r2_high_minus_low"]]
        .mean().to_string(float_format=lambda x: f"{x:+.3f}"))

print("\n(C) Zero-crossing residual ratio (RMS_near_zero / RMS_typical)")
print("Mean across joints, per task:")
print(df.groupby("task")["zerocross_ratio"]
        .agg(["mean", "min", "max"]).to_string(float_format=lambda x: f"{x:.2f}"))

print("\n(D) Off-diagonal-inertia proxy correlation |r| with residual")
print("Mean and max across joints, per task:")
df["off_diag_abs"] = df["off_diag_corr"].abs()
print(df.groupby("task")["off_diag_abs"]
        .agg(["mean", "max"]).to_string(float_format=lambda x: f"{x:.3f}"))

print("\nDone.")